In [1]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents.base import Document
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnableParallel
from dotenv import load_dotenv
import os
from euriai.langchain import create_chat_model
import time
from euriai.langchain import EuriaiEmbeddings

app_dir = os.path.join(os.getcwd(), "app")
load_dotenv(os.path.join(app_dir, ".env"))
api_key = os.getenv("key") 

loader = DirectoryLoader("./data", glob="**/*.txt")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=120,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)
chunks = text_splitter.split_documents(docs)


chat_model = create_chat_model(api_key=api_key, model="gpt-4.1-nano", temperature=0.7)
model = chat_model

embeddings = EuriaiEmbeddings(
    api_key=api_key,
    model="text-embedding-3-small"
)

db = Chroma.from_documents(chunks, embeddings)
retriever = db.as_retriever()

c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
def format_docs(docs: list[Document]):
    return "\n".join(doc.page_content for doc in docs)

In [ ]:
pipeline = (
RunnablePassthrough.assign(answer=lambda x: x["query"].upper())
)

print(pipeline.invoke({"query": "hello"}))


{'query': 'hello', 'answer': 'HELLO'}


In [ ]:
from langchain_core.runnables import RunnableLambda

template = """Answer the question based only on the following context:
{context}

Question: {question} 
"""
prompt = ChatPromptTemplate.from_template(template)

format_context = RunnableLambda(lambda x: format_docs(x["context"]))

rag_chain_from_docs = (
    RunnablePassthrough.assign(context=format_context)
    | prompt
    | model
    | StrOutputParser()
)

rag_chain_with_source = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
).assign(answer=rag_chain_from_docs)

In [10]:
result = rag_chain_with_source.invoke(input="Who is the owner of the restaurant")

In [11]:
result

{'context': [Document(id='10be78ae-8ed9-4b82-82c8-cc0b81525ee3', metadata={'source': 'data\\founder.txt'}, page_content='Creating Chef Amico’s Restaurant'),
  Document(id='f69b63a9-dc3a-4500-b605-e538b1d96ff1', metadata={'source': 'data\\founder.txt'}, page_content='craft. His spirit of generosity and passion for food extends beyond the restaurant’s walls. He mentors young chefs,'),
  Document(id='09a3526b-7c4e-46cd-bc86-16fc4f9fd05b', metadata={'source': 'data\\restaurant.txt'}, page_content="into Chef Amico. Her mission was to uncover the secret behind the restaurant's growing fame. She was greeted by Amico"),
  Document(id='774ed72a-39aa-4abf-b3bd-a5f09a481685', metadata={'source': 'data\\founder.txt'}, page_content='and relish life’s simple pleasures. His restaurant was a haven where strangers became friends over plates of arancini')],
 'question': 'Who is the owner of the restaurant',
 'answer': 'The owner of the restaurant is Chef Amico.'}

### Why is that approach bad?

You will always retrieve top-k documents and pass them to the model. 
No matter how relevant the documents are

In [12]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

ModuleNotFoundError: No module named 'sentence_transformers'

In [ ]:
docs = result["context"]
contents = [doc.page_content for doc in docs]

In [ ]:
contents

In [ ]:
pairs = []
for text in contents:
    pairs.append(["Who is the owner of the restaurant", text])

In [ ]:
pairs

In [ ]:
scores = cross_encoder.predict(pairs)
scores

In [ ]:
scored_docs = zip(scores, docs)
sorted_docs = sorted(scored_docs, reverse=True)
sorted_docs

In [ ]:
reranked_docs = [doc for _, doc in sorted_docs][0:2]
reranked_docs

### Integrate that in LCEL

In [ ]:
# retriever which retrieves more than 4 documents
retriever = db.as_retriever(search_kwargs={"k": 10})

In [ ]:
from sentence_transformers import CrossEncoder
from langchain_core.runnables import RunnableLambda


def rerank_documents(input_data):
    query = input_data["question"]
    docs = input_data["context"]

    cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    contents = [doc.page_content for doc in docs]

    pairs = [(query, text) for text in contents]
    scores = cross_encoder.predict(pairs)

    scored_docs = zip(scores, docs)
    sorted_docs = sorted(scored_docs, key=lambda x: x[0], reverse=True)
    return [doc for _, doc in sorted_docs]


template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
model = ChatOpenAI(model="gpt-4o-mini")

rag_chain_from_docs = (
    RunnablePassthrough.assign(context=RunnableLambda(rerank_documents))
    | prompt
    | model
    | StrOutputParser()
)

rag_chain_with_source = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
).assign(answer=rag_chain_from_docs)

In [ ]:
result = rag_chain_with_source.invoke(input="Who is the owner of the restaurant")
result

### LLM-based Document Compressor

In [ ]:
from langchain_core.prompts import PromptTemplate

DOCUMENT_EVALUATOR_PROMPT = PromptTemplate(
    input_variables=["document", "question"],
    template="""You are an AI language model assistant. Your task is to evaluate the provided document to determine if it is suited to answer the given user question. Assess the document for its relevance to the question, the completeness of information, and the accuracy of the content.

    Original question: {question}
    Document for Evaluation: {document}
    Evaluation Result: <<'True' if the document is suited to answer the question, 'False' if it is not>>

    Note: Conclude with a 'True' or 'False' based on your analysis of the document's relevance, completeness, and accuracy in relation to the question.""",
)

In [ ]:
from langchain_core.documents.base import Document

documents = [
    Document(page_content="The owner is Guivanni"),
    Document(page_content="Pizza Salami costs 10$"),
    Document(page_content="We close the restaurant at 10p.m each day"),
]

model = ChatOpenAI(model="gpt-4o-mini")
compression_chain = DOCUMENT_EVALUATOR_PROMPT | model | StrOutputParser()

In [ ]:
compression_chain.invoke(
    {"question": "Who is the owner of the restaurant", "document": documents[1]}
)

### Now lets make that dynamic

In [ ]:
def evaluate_documents(input: dict):
    documents = input.get("documents", [])
    question = input.get("question")

    DOCUMENT_EVALUATOR_PROMPT = PromptTemplate(
        input_variables=["document", "question"],
        template="""You are an AI language model assistant. Your task is to evaluate the provided document to determine if it is suited to answer the given user question. Assess the document for its relevance to the question, the completeness of information, and the accuracy of the content.

        Original question: {question}
        Document for Evaluation: {document}
        Evaluation Result: <<'True' if the document is suited to answer the question, 'False' if it is not>>

        Note: Conclude with a 'True' or 'False' based on your analysis of the document's relevance, completeness, and accuracy in relation to the question.""",
    )
    model = ChatOpenAI(model="gpt-4o-mini")
    compression_chain = DOCUMENT_EVALUATOR_PROMPT | model | StrOutputParser()

    results = []
    for document in documents:
        evaluation_result = compression_chain.invoke(
            {"document": document.page_content, "question": question}
        )
        result = evaluation_result == "True"
        print(result)
        results.append(result)

    filtered_documents = [doc for doc, res in zip(documents, results) if res]

    return filtered_documents

In [ ]:
_input = {
    "documents": [
        Document(page_content="The owner is Guivanni"),
        Document(page_content="Pizza Salami costs 10$"),
        Document(page_content="We close the restaurant at 10p.m each day"),
    ],
    "question": "Who is the owner of the restaurant?",
}

results = evaluate_documents(_input)
print(results)